In [26]:
import numpy as np
import random

In [27]:
# == Dataset features planned by me. Numbers made by AI.. ==
# Feature columns:
# [height (inches), weight (lbs), mile time (sec), degree level (0=None, 1=HS, 2=Bachelors, 3=Masters/PhD), can ride a bike (0=No, 1=Yes), target_label (1=baby, 2=kid, 3=teenager, 4=adult, 5=senior)]

dataset = [
    # Babies (Category 1) - Too young to ride
    [26, 18, 999, 0, 0, 1], # 1
    [30, 24, 999, 0, 0, 1], # 2
    [34, 29, 999, 0, 0, 1], # 3

    # Kids (Category 2) - Mix of younger kids (can't) and older kids (can)
    [44, 45, 720, 0, 0, 2], # 4
    [50, 60, 600, 0, 1, 2], # 5
    [56, 82, 520, 0, 1, 2], # 6

    # Teenagers (Category 3) - All can ride
    [62, 115, 450, 1, 1, 3], # 7
    [66, 135, 400, 1, 1, 3], # 8
    [70, 155, 380, 1, 1, 3], # 9

    # Adults (Category 4) - Mostly can ride, with one adult who never learned
    [64, 125, 420, 2, 1, 4], # 10
    [68, 160, 390, 2, 1, 4], # 11
    [72, 190, 360, 3, 0, 4], # 12 (Adult who never learned)
    [65, 150, 440, 2, 1, 4], # 13
    [70, 175, 410, 3, 1, 4], # 14

    # Seniors (Category 5) - Mix of active senior cyclists and those who stopped
    [62, 130, 600, 2, 1, 5], # 15
    [66, 165, 540, 3, 0, 5], # 16 (Senior who stopped riding)
    [69, 180, 720, 2, 1, 5], # 17
    [63, 140, 660, 3, 0, 5], # 18 (Senior who stopped riding)
    [67, 155, 580, 2, 1, 5], # 19
    [71, 185, 750, 3, 1, 5], # 20
]

dataset = np.asarray(dataset)

In [28]:
# pure utility. no ml related
def is_binary(X): # give a feature column
    for x in X:
        if x != 1 or x != 0:
            return False
    return True

In [71]:
class Node:
    def __init__(self, feature=None, threshold=None, left_node=None, right_node=None, *, value=None):
        self.feature = feature
        self.threshold = threshold
        self.left_node = left_node
        self.right_node = right_node
        self.value = value

    def is_leaf_node(self) -> bool:
        return self.value is not None
        
# class DecisionTree:
class DecisionTree:
    def __init__(self, max_depth=10, min_samples_split=2, min_samples_leaf=1):
        self.root_node = None
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.min_samples_leaf = min_samples_leaf
        
    def calculate_gini(self, Y):
        _, counts = np.unique(Y, return_counts=True)
        
        probs = counts / counts.sum() # normalize        
        gini = 1.0
        for p in probs:
            gini -= p**2
        return gini

    # counts the split data, according to a threshold
    def tally(self, X, feature, threshold): # given option and points
        left_mask = X[:, feature] <= threshold
        right_mask = ~left_mask
        
        return left_mask, right_mask 

    def numerical_solve(self, X, y, feature): # essentially finds the optimal midpoint for a set of numerics of a feature
        values = np.unique(X[:, feature])

        if len(values) < 2:
            return None, float("inf")
        
        midpoints = []
        for i in range(len(values) - 1):
            midpoint = (values[i] + values[i+1]) / 2
            midpoints.append(midpoint)
    
        lowest_impurity = float("inf") # 0 is the best
        best_midpoint = None
        
        for midpoint in midpoints: # tally the numbers
            left_mask, right_mask = self.tally(X, feature, midpoint)

            y_left = y[left_mask]
            y_right = y[right_mask]

            n_left = len(y_left)
            n_right = len(y_right)
            n_total = len(y)

            # reject splits creating excessively small leaves
            if (n_left < self.min_samples_leaf or n_right < self.min_samples_leaf):
                continue
                
            # Weighted impurity of both children
            gini_impurity = (
                 (n_left / n_total) * self.calculate_gini(y_left)
                + (n_right / n_total) * self.calculate_gini(y_right)
            )
            
            if gini_impurity < lowest_impurity:
                lowest_impurity = gini_impurity
                best_midpoint = midpoint
    
        return best_midpoint, lowest_impurity

    def pass_data(X): # returns either the resulting leaf node, a node that returns None for the next node
        current_node = self.root_node
        while True: # loops down the nodes, following the data's features.
            if current_node.is_leaf_node(): return current_node # if leaf node, then return the result.
            if X[current_node.feature] > current_node.value:
                if current_node.right_node == None:
                    return current_node # dead end, not fully built
                else: 
                    current_node = current_node.right_node # pass next node
            else: # go to left node
                if current_node.left_node == None:
                    return current_node # dead end, not fully built
                else:
                    current_node = current_node.left_node # pass next node

        raise Value("Broken Data Pass")

    
    def build_first_node(self, X, y):
        lowest_impurity = float("inf")
        best_threshold = None
        best_feature = None
        
        for feature_i in range(X.shape[1]):
            midpoint, impurity = self.numerical_solve(X, y, feature_i)
            if midpoint is not None and impurity < lowest_impurity:
                lowest_impurity = impurity
                best_threshold = midpoint
                best_feature = feature_i

        if best_feature is None:
            return None, float("inf")

        node = Node(
            feature=best_feature,
            threshold=best_threshold
        )
        
        return node, lowest_impurity

    def most_common_label(self, y):
        labels, counts = np.unique(y, return_counts=True)
        return labels[np.argmax(counts)]

    def build_tree(self, X, y, depth=0):
        n_samples = len(y)
        n_classes = len(np.unique(y))

        if (
            depth >= self.max_depth
            or n_samples < self.min_samples_split
            or n_classes == 1
        ):
            return Node(value=self.most_common_label(y))

        # find the best possible split for this node
        node, child_impurity = self.build_first_node(X, y)

        # stop if no valid or useful split exists
        parent_impurity = self.calculate_gini(y)

        if (node is None or parent_impurity - child_impurity <= 1e-12): \
            return Node(value=self.most_common_label(y))

        # split training samples
        left_mask, right_mask = self.tally(
            X, node.feature, node.threshold
        )

        X_left = X[left_mask]
        y_left = y[left_mask]

        X_right = X[right_mask]
        y_right = y[right_mask]

        # recursively construct both children
        node.left_node = self.build_tree(
            X_left, y_left, depth + 1
        )

        node.right_node = self.build_tree(
            X_right, y_right, depth + 1
        )

        return node

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y)

        if X.ndim != 2:
            raise ValueError("X must be a 2D array")

        if y.ndim != 1 or len(y) != len(X) or len(y) == 0:
            raise ValueError("Invalid training labels")

        if not np.all(np.isfinite(X)):
            raise ValueError("X must contain finite numbers")

        self.root_node = self.build_tree(X, y)
        return self

    def pass_data(self, X):
        current_node = self.root_node

        if current_node == None:
            raise ValueError("Tree has not been trained")

        while not current_node.is_leaf_node():
            if X[current_node.feature] <= current_node.threshold:
                current_node = current_node.left_node
            else:
                current_node = current_node.right_node

        return current_node

    def predict(self, X):
        X = np.asarray(X, dtype=float)

        if X.ndim == 1:
            X = X.reshape(1, -1)

        predictions = []

        for row in X:
            leaf = self.pass_data(row)
            predictions.append(leaf.value)

        return np.array(predictions)

    def print_tree(self, node=None, depth=0):
        if node is None:
            node = self.root_node
        if node is None:
            raise ValueError("Tree has not been trained")

        indent = "   " * depth
        if node.is_leaf_node():
            print(f"{indent}Predict: {node.value}")
            return
            
        print(
            f"{indent}if X[{node.feature}] <= "
            f"{node.threshold:.4f}:"
        )
        self.print_tree(node.left_node, depth + 1)

        print(f"{indent}else:")
        self.print_tree(node.right_node, depth + 1)

In [65]:
X = dataset[:, :-1]
y = dataset[:, -1:]
print(y.shape)
# print("X: ", X)
# print("y: ", y)
# print(len(X) == len(y))

(20, 1)


In [72]:
X = dataset[:, :-1]
y = dataset[:, -1:].squeeze(axis=-1)

tree = DecisionTree(max_depth=5)
tree.fit(X, y)
tree.print_tree()

print("Predicitons: ")
print(tree.predict([
    [41, 52, 695, 0, 1], # kid. Answer is 2
    [60, 134, 610, 2, 0], # senior Answer is 5
    ])
)

if X[2] <= 485.0000:
   if X[3] <= 1.5000:
      Predict: 3
   else:
      Predict: 4
else:
   if X[0] <= 59.0000:
      if X[0] <= 39.0000:
         Predict: 1
      else:
         Predict: 2
   else:
      Predict: 5
Predicitons: 
[2 5]


In [4]:
# Random Forest
# TODO: Build a random Forest

In [5]:
# XGBoost Random Forest
# TODO: Build an extreme gradient boosted version of random forest